# Lagrangian Displacement SR — Validation Notebook

**Goal**: validate the displacement super-resolution pipeline step by step.

| Section | Question |
|---------|----------|
| 1 | Is ΔΨ well-defined and non-trivial? |
| 2 | Does ΔΨ correlate with Lagrangian features? |
| 3 | Training monitoring |
| 4 | Inference: does the MLP predict ΔΨ? |
| 5 | Spatial validation + shell-crossing |
| 6 | Generalisation across snapshots |
| 7 | Corrected positions vs HR positions |

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import yaml
from types import SimpleNamespace
from pathlib import Path
import pickle
import numpy as np
import jax
import jax.numpy as jnp
import haiku as hk
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.stats import pearsonr

import sys
REPO_ROOT = Path("../").resolve()   # adjust if needed
sys.path.insert(0, str(REPO_ROOT / "pm2nbody"))

from train_lag_disp import (
    get_lagrangian_positions,
    compute_displacement_pair,
    compute_cnn_disp_correction,
    compute_disp_metrics,
)
from train_lag_force import (
    load_snapshot,
    snapshot_features,
)
from jaxpm.lagrangian import (
    get_axis_neighbor_indices,
    get_shell_neighbor_indices,
    make_lagrangian_corrector,
)
print("imports OK  |  JAX devices:", jax.devices())

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CONFIGURATION — edit this cell
# ═══════════════════════════════════════════════════════════════════
CONFIG_PATH = REPO_ROOT / "configs/lag_disp_mlp.yaml"
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

data_cfg  = SimpleNamespace(**cfg["data"])
model_cfg = SimpleNamespace(**cfg["model"])

DATA_DIR   = Path(data_cfg.data_dir)
MESH_LR    = int(data_cfg.mesh_lr)
MESH_HR    = int(data_cfg.mesh_hr)
BOX_SIZE   = float(data_cfg.box_size)
N_PART     = int(data_cfg.n_particles)

SIM_TRAIN  = int(data_cfg.sim_id_train)
SIM_VAL    = int(getattr(data_cfg, "sim_id_val", 1))
SNAP_TRAIN = int(data_cfg.snap_train)
SNAP_VAL   = int(data_cfg.snap_train)
SNAPS_VAL  = list(data_cfg.snaps_val)

USE_STRAIN     = bool(model_cfg.use_strain)
USE_INVARIANTS = bool(model_cfg.use_invariants)
USE_VELOCITY   = bool(model_cfg.use_velocity)
N_SHELL        = int(getattr(model_cfg, "n_shell", 0))
ENV_POOL_MODE  = str(getattr(model_cfg, "env_pool_mode", "mean_var"))

CHECKPOINT_DIR = REPO_ROOT / "runs/lag_disp"
SV_CENTER = [MESH_LR // 2] * 3
SV_HALF   = max(6, MESH_LR // 10)

print(f"mesh_lr={MESH_LR}  mesh_hr={MESH_HR}  box={BOX_SIZE} Mpc/h")
print(f"train sim={SIM_TRAIN} snap={SNAP_TRAIN} | val sim={SIM_VAL} snaps={SNAPS_VAL}")
print(f"n_shell={N_SHELL}  env_pool_mode='{ENV_POOL_MODE}'")

In [ ]:
# ── Lagrangian neighbour indices (shared across sections) ──────────────────
neighbor_idx = get_axis_neighbor_indices(N_PART)

ext_neighbor_idx, ext_offsets, shell_slices = None, None, ()
if N_SHELL > 0:
    ext_neighbor_idx, ext_offsets, shell_slices = get_shell_neighbor_indices(
        N_PART, N_SHELL
    )
    K = (2 * N_SHELL + 1)**3 - 1
    print(f"Extended neighbourhood: n_shell={N_SHELL}  K={K}  pool='{ENV_POOL_MODE}'")
else:
    print("No extended neighbourhood (n_shell=0)")


def get_feats(pos_t):
    """Compute (feats, det_D) using the same settings as during training."""
    return snapshot_features(
        pos_t, neighbor_idx, MESH_LR, USE_STRAIN, USE_INVARIANTS,
        ext_neighbor_idx, ext_offsets, ENV_POOL_MODE, shell_slices,
    )

print("Neighbour indices ready.")

## Section 1 — Displacement target: sanity & signal

In [ ]:
pos_lr_t, vel_lr_t, pos_hr_t, a_train = load_snapshot(
    DATA_DIR, SIM_TRAIN, SNAP_TRAIN, MESH_LR, MESH_HR, BOX_SIZE
)
assert pos_hr_t is not None, "HR positions not found — check DATA_DIR and MESH_HR"
print(f"a_train={a_train:.4f}  N_LR={pos_lr_t.shape[0]:,}  N_HR={pos_hr_t.shape[0]:,}")

feats_train, det_D_train = get_feats(pos_lr_t)
feat_dim_train = int(feats_train.shape[1])
strain_mag_train = np.sqrt(np.sum(np.asarray(feats_train[:, :9])**2, axis=-1))
sc_frac_train    = float(np.mean(det_D_train < 0))

_disp_pair = jax.jit(lambda pl, ph: compute_displacement_pair(pl, ph, N_PART, MESH_LR, MESH_HR))
psi_lr_tr, psi_hr_tr, delta_psi_tr = _disp_pair(pos_lr_t, pos_hr_t)

dpsi_np    = np.asarray(jax.device_get(delta_psi_tr))
psi_hr_np  = np.asarray(jax.device_get(psi_hr_tr))
psi_lr_np  = np.asarray(jax.device_get(psi_lr_tr))
dpsi_mag   = np.sqrt(np.sum(dpsi_np**2,  axis=-1))
psi_hr_mag = np.sqrt(np.sum(psi_hr_np**2, axis=-1))

print(f"\n|Psi_LR| mean = {np.sqrt(np.sum(psi_lr_np**2,axis=-1)).mean():.4e}")
print(f"|Psi_HR| mean = {psi_hr_mag.mean():.4e}")
print(f"|DPsi|   mean = {dpsi_mag.mean():.4e}  ({dpsi_mag.mean()/psi_hr_mag.mean():.1%} of HR)")
print(f"SC frac = {sc_frac_train:.3%}")
print(f"Feature dim = {feat_dim_train}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ss = np.random.default_rng(0).choice(len(dpsi_mag), min(50_000, len(dpsi_mag)), replace=False)

# ΔΨ component distribution
for ci, (comp, color) in enumerate(zip(["x","y","z"],["C0","C1","C2"])):
    axes[0].hist(dpsi_np[:, ci], bins=200, alpha=0.6, color=color,
                 density=True, label=f"dpsi_{comp}")
axes[0].set_xlabel("dpsi [mesh_lr units]"); axes[0].legend()
axes[0].set_title("Displacement correction distribution")

# |ΔΨ| vs |Ψ_HR|
axes[1].hexbin(psi_hr_mag[ss], dpsi_mag[ss], gridsize=80,
               norm=LogNorm(), mincnt=1, cmap="plasma")
axes[1].set_xlabel("|Psi_HR| [mesh]"); axes[1].set_ylabel("|DPsi|")
axes[1].set_title(f"|DPsi| vs |Psi_HR|  a={a_train:.3f}")

# |ΔΨ| vs strain
r_strain, _ = pearsonr(strain_mag_train[ss], dpsi_mag[ss])
axes[2].hexbin(strain_mag_train[ss], dpsi_mag[ss], gridsize=80,
               norm=LogNorm(), mincnt=1, cmap="viridis")
axes[2].set_xlabel("||E||_F"); axes[2].set_ylabel("|DPsi|")
axes[2].set_title(f"R(strain, |DPsi|) = {r_strain:.4f}")

plt.suptitle(f"Displacement correction  a={a_train:.3f}", fontsize=12)
plt.tight_layout(); plt.show()
print("Signal verdict:", "PASS" if abs(r_strain) > 0.2 else "WEAK" if abs(r_strain) > 0.05 else "FAIL")

## Section 2 — Feature–displacement correlation

In [ ]:
feat_names = [f"E_{ab}" for ab in ["xx","xy","xz","yx","yy","yz","zx","zy","zz"]]
if USE_INVARIANTS:
    feat_names += ["tr(D)", "det(D)", "||E||_F"]
if N_SHELL > 0:
    n_env = feat_dim_train - len(feat_names)
    feat_names += [f"env_{i}" for i in range(n_env)]

feats_np = np.asarray(jax.device_get(feats_train))
n_feat   = feats_np.shape[1]
names_to_plot = feat_names[:n_feat]

r_vals = [pearsonr(feats_np[:, fi], dpsi_mag)[0] for fi in range(n_feat)]

fig, ax = plt.subplots(figsize=(max(10, n_feat * 0.7), 4))
colors  = ["tomato" if abs(r) > 0.1 else "steelblue" for r in r_vals]
ax.bar(range(len(r_vals)), r_vals, color=colors)
ax.set_xticks(range(len(r_vals)))
ax.set_xticklabels(names_to_plot, rotation=45, ha="right")
ax.axhline(0, color="k", lw=0.5)
ax.set_ylabel("Pearson R  with  |DPsi|")
ax.set_title(f"Feature-displacement correlation  (a={a_train:.3f})  red = |R|>0.1")
plt.tight_layout(); plt.show()

## Section 3 — Training monitoring

In [ ]:
history = None
try:
    import wandb
    api  = wandb.Api()
    runs = api.runs("pm2nbody_lag_disp", order="-created_at", per_page=1)
    run  = next(iter(runs))
    history = run.history(samples=2000)
    print(f"WandB run: {run.name}  ({len(history)} steps)")
except Exception as e:
    print(f"WandB not available: {e}")

if history is not None:
    import pandas as pd
    df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    if "train/loss"           in df: axes[0].semilogy(df["_step"], df["train/loss"]);           axes[0].set_title("Train loss")
    if "train/pearson_r_mean" in df: axes[1].plot(df["_step"], df["train/pearson_r_mean"]);     axes[1].set_title("Train R")
    if "val/disp_mse_mean"    in df: axes[2].semilogy(df["_step"], df["val/disp_mse_mean"]);    axes[2].set_title("Val MSE")
    plt.tight_layout(); plt.show()

## Section 4 — Inference

In [ ]:
PARAMS_LOADED = None

run_dirs = sorted(CHECKPOINT_DIR.glob("*/"), key=lambda p: p.stat().st_mtime, reverse=True)
for rd in run_dirs:
    for fname in ["best_params.pkl", "final_params.pkl"]:
        pkl = rd / fname
        if pkl.exists():
            with open(pkl, "rb") as fh:
                PARAMS_LOADED = hk.data_structures.to_immutable_dict(pickle.load(fh))
            print(f"Loaded: {pkl}"); break
    if PARAMS_LOADED is not None: break

if PARAMS_LOADED is None:
    print("No checkpoint found in", CHECKPOINT_DIR)

# Build model with SAME hidden_dim, n_layers, output_dim=3 as training
lag_model = make_lagrangian_corrector(
    hidden_dim=int(model_cfg.hidden_dim),
    n_layers=int(model_cfg.n_layers),
    output_dim=3,
)

# Validation snapshot
pos_v, vel_v, pos_hr_v, a_val = load_snapshot(
    DATA_DIR, SIM_VAL, SNAP_VAL, MESH_LR, MESH_HR, BOX_SIZE
)
# Use get_feats() — includes extended neighbourhood if n_shell > 0
feats_v, det_D_v = get_feats(pos_v)
feat_dim_val     = int(feats_v.shape[1])
strain_mag_v     = np.sqrt(np.sum(np.asarray(feats_v[:, :9])**2, axis=-1))
sc_frac_v        = float(np.mean(det_D_v < 0))

_, _, delta_psi_v = _disp_pair(pos_v, pos_hr_v)
dpsi_v_np  = np.asarray(jax.device_get(delta_psi_v))
dpsi_v_mag = np.sqrt(np.sum(dpsi_v_np**2, axis=-1))
vel_feat_v = vel_v if USE_VELOCITY else jnp.zeros_like(vel_v)

print(f"Validation  a={a_val:.4f}  SC={sc_frac_v:.2%}  |DPsi| mean={dpsi_v_mag.mean():.4e}")
print(f"Feature dim val = {feat_dim_val}  (train = {feat_dim_train})")
assert feat_dim_val == feat_dim_train, (
    f"Feature dim mismatch: val={feat_dim_val} != train={feat_dim_train}. "
    "Check n_shell / env_pool_mode in config."
)

In [ ]:
if PARAMS_LOADED is None:
    print("No checkpoint.")
else:
    pred_v_np = np.asarray(jax.device_get(
        jax.jit(lag_model.apply)(PARAMS_LOADED, feats_v, vel_feat_v, jnp.array(a_val))
    ))
    err      = pred_v_np - dpsi_v_np
    mse_tot  = float(np.mean(err**2))
    frac_mse = float(np.mean(np.sum(err**2, axis=1) / (np.sum(dpsi_v_np**2, axis=1) + 1e-12)))
    r_xyz    = [pearsonr(pred_v_np[:, c], dpsi_v_np[:, c])[0] for c in range(3)]
    r_mean   = float(np.mean(r_xyz))

    print(f"MSE={mse_tot:.4e}  FracMSE={frac_mse:.4f}  R(x,y,z)=({r_xyz[0]:.3f},{r_xyz[1]:.3f},{r_xyz[2]:.3f})  R_mean={r_mean:.4f}")

    COMPS = ["x","y","z"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    ss = np.random.default_rng(42).choice(len(pred_v_np), min(40_000, len(pred_v_np)), replace=False)
    for ci in range(3):
        tgt  = dpsi_v_np[ss, ci]; pred = pred_v_np[ss, ci]
        lim  = max(abs(np.percentile(tgt, 1)), abs(np.percentile(tgt, 99))) * 1.15
        ax   = axes[ci]
        h    = ax.hexbin(tgt, pred, gridsize=70, cmap="Blues",
                         norm=LogNorm(), mincnt=1, extent=[-lim,lim,-lim,lim])
        plt.colorbar(h, ax=ax)
        ax.plot([-lim,lim],[-lim,lim],"r--",lw=1)
        ax.set_xlabel(f"Target dPsi_{COMPS[ci]}")
        ax.set_ylabel(f"Pred  dPsi_{COMPS[ci]}")
        ax.set_title(f"{COMPS[ci]}  R={pearsonr(tgt,pred)[0]:.4f}")
    plt.suptitle(f"dPsi pred vs target  a={a_val:.3f}", fontsize=12)
    plt.tight_layout(); plt.show()

## Section 5 — Spatial validation + corrected positions

In [ ]:
if PARAMS_LOADED is not None:
    pos_v_np   = np.asarray(jax.device_get(pos_v)) % MESH_LR
    sv_mask    = np.all(np.abs(pos_v_np - np.array(SV_CENTER)) < SV_HALF, axis=1)
    idx_sv     = np.where(sv_mask)[0]
    px, py     = pos_v_np[idx_sv, 0], pos_v_np[idx_sv, 1]

    # Corrected & HR positions
    pos_v_full = np.asarray(jax.device_get(pos_v))
    pos_hr_np  = np.asarray(jax.device_get(pos_hr_v))
    stride     = MESH_HR // N_PART
    x_hr_at_q  = pos_hr_np.reshape(MESH_HR, MESH_HR, MESH_HR, 3)[::stride, ::stride, ::stride, :].reshape(-1,3)
    x_corrected = pos_v_full + pred_v_np

    # Sub-volume: |ΔΨ| target / pred / error
    pred_mag = np.sqrt(np.sum(pred_v_np**2, axis=-1))
    err_mag  = np.sqrt(np.sum((pred_v_np - dpsi_v_np)**2, axis=-1))
    vmax     = np.percentile(dpsi_v_mag, 95)
    panels   = [
        (dpsi_v_mag, "|dPsi_target|", "Oranges"),
        (pred_mag,   "|dPsi_pred|",   "Blues"),
        (err_mag,    "|error|",        "hot"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ci, (mag, label, cmap) in enumerate(panels):
        ax = axes[ci]
        hb = ax.hexbin(px, py, C=mag[idx_sv], gridsize=60, cmap=cmap,
                       reduce_C_function=np.mean, vmin=0,
                       vmax=vmax if ci < 2 else np.percentile(mag, 95))
        plt.colorbar(hb, ax=ax, label=label)
        ax.set_title(label); ax.set_xlabel("x")
    plt.suptitle(f"Sub-volume  a={a_val:.3f}  SC={sc_frac_v:.2%}", fontsize=11)
    plt.tight_layout(); plt.show()

    # Position maps: LR / corrected / HR
    labels = ["LR positions", "LR + MLP", "HR reference"]
    cmaps  = ["Blues", "Greens", "Reds"]
    xs     = [
        pos_v_np[idx_sv],
        x_corrected[idx_sv] % MESH_LR,
        x_hr_at_q[idx_sv]   % MESH_LR,
    ]
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ci, (x, label, cmap) in enumerate(zip(xs, labels, cmaps)):
        ax = axes[ci]
        hb = ax.hexbin(x[:, 0], x[:, 1], gridsize=70, cmap=cmap, norm=LogNorm(), mincnt=1)
        plt.colorbar(hb, ax=ax, label="counts")
        ax.set_title(label); ax.set_xlabel("x")

    err_lr  = np.sqrt(np.sum((pos_v_full - x_hr_at_q)**2, axis=-1)).mean()
    err_cor = np.sqrt(np.sum((x_corrected - x_hr_at_q)**2, axis=-1)).mean()
    improv  = 1.0 - err_cor / err_lr
    plt.suptitle(f"|err_LR|={err_lr:.4e}  |err_corrected|={err_cor:.4e}  improvement={improv:.1%}", fontsize=10)
    plt.tight_layout(); plt.show()
    print(f"Position improvement: {improv:.1%}")

## Section 6 — Generalisation across snapshots

In [ ]:
if PARAMS_LOADED is not None:
    import pandas as pd
    rows = []
    for vsnap in SNAPS_VAL:
        vpos, vvel, vpos_hr_s, va = load_snapshot(DATA_DIR, SIM_VAL, vsnap, MESH_LR, MESH_HR, BOX_SIZE)
        vfeats, vdet_D = get_feats(vpos)   # ← uses extended neighbourhood if n_shell>0
        _, _, vdpsi    = _disp_pair(vpos, vpos_hr_s)
        vvel_feat = vvel if USE_VELOCITY else jnp.zeros_like(vvel)
        vpred     = jax.jit(lag_model.apply)(PARAMS_LOADED, vfeats, vvel_feat, jnp.array(va))

        vpos_np  = np.asarray(jax.device_get(vpos))
        vpred_np = np.asarray(jax.device_get(vpred))
        vdpsi_np = np.asarray(jax.device_get(vdpsi))
        stride_v = MESH_HR // N_PART
        vhr_grid = np.asarray(jax.device_get(vpos_hr_s)).reshape(MESH_HR, MESH_HR, MESH_HR, 3)
        vx_hr    = vhr_grid[::stride_v, ::stride_v, ::stride_v, :].reshape(-1, 3)

        err_lr_v  = np.sqrt(np.sum((vpos_np - vx_hr)**2, axis=-1)).mean()
        err_cor_v = np.sqrt(np.sum((vpos_np + vpred_np - vx_hr)**2, axis=-1)).mean()
        met       = compute_disp_metrics(vpred, vdpsi, vdet_D)

        rows.append({"snap": vsnap, "a": float(va),
                     "R_mean": met["pearson_r_mean"], "frac_mse": met["frac_mse"],
                     "err_lr": err_lr_v, "err_cor": err_cor_v,
                     "improvement_%": (1.0 - err_cor_v / err_lr_v) * 100})
        del vpos, vvel, vfeats, vdpsi, vpred

    df_res = pd.DataFrame(rows)
    print(df_res.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].plot(df_res["a"], df_res["R_mean"],        "o-"); axes[0].set_title("R_mean vs a")
    axes[1].semilogy(df_res["a"], df_res["frac_mse"],  "o-"); axes[1].set_title("FracMSE vs a")
    axes[2].plot(df_res["a"], df_res["improvement_%"], "o-"); axes[2].set_title("Position improvement % vs a")
    axes[2].axhline(0, color="k", lw=0.5)
    for ax in axes: ax.set_xlabel("scale factor a")
    plt.tight_layout(); plt.show()

## Section 7 — Verdict

In [ ]:
if PARAMS_LOADED is not None:
    print("="*60)
    print("  LAGRANGIAN DISPLACEMENT SR — VERDICT")
    print("="*60)
    print(f"  R_mean (val)     : {r_mean:.4f}")
    print(f"  FracMSE (val)    : {frac_mse:.4f}")
    print(f"  Pos improvement  : {improv:.1%}")
    if "df_res" in dir():
        print(f"  R across snaps   : {list(df_res.R_mean.round(3).values)}")
        print(f"  Improvement % s  : {list(df_res['improvement_%'].round(1).values)}")
    print()
    checks = [
        (r_mean > 0.5,  f"R_mean={r_mean:.3f} > 0.5  (displacement predictive)"),
        (frac_mse < 0.5, f"FracMSE={frac_mse:.3f} < 0.5"),
        (improv > 0,    f"position improves by {improv:.1%}"),
    ]
    for ok, msg in checks:
        print(f"  {'OK' if ok else 'XX'} {msg}")
    print("="*60)